# 04 · Contradiction analysis

What the contradiction stage (A3) did on each claim: its initial verdict (= A2), the challenges it raised, the counter-evidence retrieved, and whether the final verdict changed — compared with the self-reflection control (SR), which gets the same number of extra model calls but no new evidence. Correctness of changes is shown only for claims with finalized gold labels.

In [ ]:
import os, sys, json
from dataclasses import asdict
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
print("research root:", ROOT)

In [ ]:
RUN_ID = None
EXPERIMENT_ID = "claimverif-main-v1"

In [ ]:
from engine.notebook import banner, pick_run, describe_run, load_gold_labels, trace_index, save_figure
run = pick_run(EXPERIMENT_ID, RUN_ID)
assert run is not None, "no run found -- execute notebook 03 first"
describe_run(run)
gold = load_gold_labels()
if not gold:
    banner("No finalized gold labels: the help / harm / false-contradiction columns cannot be computed yet and are left empty.")
preds = pd.DataFrame(run["predictions"])
traces = trace_index(run)

## A3: initial vs final verdict, and what the investigation found

In [ ]:
def second_look_rows(kind, key):
    rows = []
    for p in run["predictions"]:
        if p.get("approach_kind") != kind:
            continue
        info = p.get(key) or {}
        g = gold.get(p["claim_id"], {}).get("label")
        rows.append({"approach": p["approach"], "claim_id": p["claim_id"], "category": p["claim_category"],
                     "initial": p.get("initial_label"), "final": p.get("predicted_label"), "changed": p.get("verdict_changed"),
                     "items": len(info.get("hypotheses") or info.get("weaknesses") or []),
                     "counter_passages": len(info.get("counter_evidence_ids") or []),
                     "counter_numeric": len(info.get("counter_numeric") or []),
                     "challenges_supported": info.get("challenges_supported"), "invalid_challenges": info.get("invalid_challenges"),
                     "guard_reverted": info.get("guard_reverted"), "numeric_rule": info.get("final_numeric_rule"),
                     "gold": g, "initial_correct": (p.get("initial_label") == g) if g else None,
                     "final_correct": (p.get("predicted_label") == g) if g else None})
    return pd.DataFrame(rows)
a3 = second_look_rows("contradiction_aware", "contradiction")
sr = second_look_rows("self_reflection", "reflection")
a3

## Verdict transitions

In [ ]:
for name, d in (("A3 contradiction-aware", a3), ("SR self-reflection", sr)):
    if d.empty:
        print(name, ": not in this run"); continue
    print(f"{name}: changed {int(d.changed.sum())}/{len(d)} verdicts; guard reverted {d.guard_reverted.notna().sum()}; "
          f"challenges asserted without valid evidence: {int(d.invalid_challenges.fillna(0).sum())}")
    display(pd.crosstab(d.initial, d.final, margins=True))

## Did changes help or harm? (finalized gold only)

In [ ]:
def outcome(r):
    if r.gold is None or pd.isna(r.gold):
        return None
    if not r.changed:
        return "unchanged"
    return {(False, True): "helped", (True, False): "harmed", (False, False): "changed, still wrong"}.get((r.initial_correct, r.final_correct), "changed")
for name, d in (("A3", a3), ("SR", sr)):
    if d.empty:
        continue
    d["outcome"] = d.apply(outcome, axis=1)
    labelled = d[d.gold.notna()]
    if labelled.empty:
        print(name, ": no labelled claims"); continue
    false_contra = labelled[(labelled.final == "CONTRADICTED") & (labelled.gold != "CONTRADICTED")]
    print(f"{name}: {labelled.outcome.value_counts().to_dict()} | false contradictions: {len(false_contra)}")
    display(pd.crosstab(labelled.category, labelled.outcome))

## By claim category

In [ ]:
cat = pd.concat([d.assign(method=n) for n, d in (("A3", a3), ("SR", sr)) if not d.empty])
if not cat.empty:
    summary = cat.groupby(["method", "category"]).agg(claims=("claim_id", "count"), change_rate=("changed", "mean"),
                                                      avg_counter_passages=("counter_passages", "mean"))
    display(summary.round(2))
    fig, ax = plt.subplots(figsize=(6, 2.8))
    summary.change_rate.unstack(0).plot.bar(ax=ax, rot=0, title="Share of verdicts changed by the second look")
    ax.spines[["top", "right"]].set_visible(False); plt.tight_layout()
    print("saved", save_figure(fig, f"04_change_rate_{run['manifest']['run_id']}", run))

## Investigation examples

Challenges, queries and the counter-evidence actually retrieved (changed verdicts first).

In [ ]:
ex = a3.sort_values("changed", ascending=False).head(3)
for _, r in ex.iterrows():
    t = traces.get((r.approach, r.claim_id)) or {}
    print("=" * 100, f"\n{r.claim_id}  {r.initial} -> {r.final}   claim: {t.get('claim', {}).get('claim')}")
    inv = next((s for s in t.get("stages", []) if s["stage"] == "contradiction_investigation"), {})
    ev = {e["evidence_id"]: e for e in t.get("evidence", [])}
    for h in inv.get("hypotheses", []):
        print(f"  [{h['hypothesis_id']}] {h['challenge']}\n      query: {h.get('search_query')}")
    for q in inv.get("queries", []):
        for eid in q["results"][:1]:
            e = ev.get(eid, {})
            print(f"      {q['hypothesis_id']} -> {eid} {e.get('form')} filed {e.get('filed')}: {(e.get('excerpt') or '')[:240]}…")
    rec = next((s for s in t.get("stages", []) if s["stage"] == "reconciliation"), {})
    for a in rec.get("assessments", []):
        print(f"  assessment {a['item_id']}: {a['assessment']}" + (f" (declared {a['declared']}, {a['flag']})" if a.get("flag") else ""))